# Stage 4 — Evaluation Pipeline

**Project:** LLM-Based Information Extraction from Hospitality Request Emails: A Prompt Engineering Approach  
**Author:** Mikail Imran Shaukat

---

## What this notebook does

This notebook evaluates the extraction results from Stage 3 against a manually annotated ground truth dataset.

Each of the 9 pipeline configurations (3 prompt variants × 3 model variants) are evaluated at both the overall level and the field level using precision, recall, and F1 score. A mismatch analysis is also included to inspect specific false positive and false negative cases per field.

**Input:** `Stage 3 - Extraction Results.csv` — 450 extraction results from Stage 3  
**Input:** `Stage 4 - Ground Truth Annotations for Evaluation.xlsx` — manually annotated ground truth for all 50 emails  
**Output:** `Stage 4 - Overall Evaluation.csv` — overall precision, recall, and F1 per configuration (9 rows)  
**Output:** `Stage 4 - Field Level Evaluation.csv` — field-level precision, recall, and F1 per configuration (108 rows)

---
## Cell 1 — Setup and Load Data

The two input files are loaded here. The extraction results contain 450 rows (one per email per configuration). The ground truth annotations are in long format with 600 rows (one per field per email).

The shape of both dataframes is printed as a quick check to confirm the files loaded correctly before anything else runs.

In [2]:
import pandas as pd
import numpy as np
from itertools import product

df_results = pd.read_csv('Stage 3 - Extraction Results.csv')
df_gt      = pd.read_excel('Stage 4 - Ground Truth Annotations for Evaluation.xlsx', sheet_name='Ground Truth Annotations', header = 3)

print(f'Extraction results: {df_results.shape}')
print(f'Ground truth:       {df_gt.shape}')
print()
print(df_results.head(3))
print()
print(df_gt.head(3))

Extraction results: (450, 17)
Ground truth:       (600, 3)

  prompt_version model_name  email_id           event_date     event_type  \
0             v1     gpt-4o         1  Thursday, March 5th       workshop   
1             v1     gpt-4o         2   Monday, April 14th      tech talk   
2             v1     gpt-4o         3   Wednesday, July 22  partner event   

            event_time                    meal_type attendee_count  \
0  10:00 AM to 3:00 PM  light lunch, coffee and tea             18   
1              1:00 PM     pizzas or something easy             25   
2   2:00 PM to 6:00 PM      refreshments and snacks             15   

  external_guests catering_type office_location dietary_requirements  \
0             Yes           NaN             NaN                  NaN   
1             NaN           NaN   Wibaut office                  NaN   
2             NaN           NaN             NaN                  NaN   

  room_location                         av_equipment  \
0   O

---
## Cell 2 — Define Fields, Configurations, and Pivot Ground Truth

The 12 fields, 3 prompt versions, and 3 models are defined here as lists to control the order in which results are displayed throughout the notebook.

The ground truth is then pivoted from long format to wide format. In long format there is one row per field per email (600 rows total). In wide format there is one row per email with one column per field (50 rows total). The pivot makes it straightforward to look up the ground truth value for any field for any email in a single operation during evaluation.

The `GT_NULL` constant defines how absent fields are represented in the ground truth as the string `"Not mentioned"` since all annotations were entered manually.

Finally, the number of GT positives per field and in total are pre-computed here. This are used as the recall denominator in `compute_metrics` to ensure recall reflects all ground truth positives.

In [3]:
FIELDS  = [
    "event_date", "event_type", "event_time", "meal_type",
    "attendee_count", "external_guests", "catering_type",
    "office_location", "dietary_requirements", "room_location",
    "av_equipment", "extra_arrangements"
]

PROMPTS = ["v1", "v2", "v3"]
MODELS  = ["gpt-4o", "gpt-4o-mini", "gpt-4.1-nano"]

GT_NULL = "Not mentioned"

df_gt_wide = df_gt.pivot(
    index='Email ID',
    columns='Field',
    values='Ground Truth Value'
).reset_index()

# A GT positive is any annotation that is not "Not mentioned"
gt_positives_per_field = {
    field: (df_gt_wide[field] != GT_NULL).sum()
    for field in FIELDS
}
total_gt_positives = sum(gt_positives_per_field.values())

print(f'Ground truth wide format: {df_gt_wide.shape}')
print()
print(f'GT positives per field:')
for field, count in gt_positives_per_field.items():
    print(f'  {field:<25} {count}')
print(f'  {"TOTAL":<25} {total_gt_positives}')

Ground truth wide format: (50, 13)

GT positives per field:
  event_date                50
  event_type                50
  event_time                36
  meal_type                 35
  attendee_count            48
  external_guests           34
  catering_type             27
  office_location           24
  dietary_requirements      25
  room_location             45
  av_equipment              8
  extra_arrangements        23
  TOTAL                     405


---
## Cell 3 — Matching Functions

Two functions handle the comparison logic used throughout the evaluation.

**`is_null(value)`**  
Checks whether a value represents an absent field. The ground truth annotations use the string `"Not mentioned"` for absent fields since all annotations were entered manually. However the extraction results can represent null in several different ways such as Python `None`, pandas `NaN` (which appears when loading null values from CSV), empty strings, or the literal string `"null"`. This function handles all cases.

**`is_match(extracted, ground_truth)`**  
Determines whether an extracted value is a correct match for the ground truth. Exact string matching was considered at first but rejected because slight differences between the extracted value and the ground truth such as verbosity differences or minor phrasing variations would cause semantically correct extractions to be scored as incorrect. For example, the ground truth `"Microsoft team"` and the extracted value `"Microsoft"` are semantically equivalent but would fail an exact match.

A token-level overlap matching approach was adopted instead. The function removes common stop words from both values, then scores a match as correct if the meaningful word overlap covers at least 50% of the ground truth words. This threshold was chosen to be strict enough to penalise genuinely wrong extractions while allowing for natural variation in phrasing.

Sanity checks are printed at the end to confirm both functions behave as expected before the evaluation runs.

In [4]:
def is_null(value):
    if value is None:
        return True
    if isinstance(value, float) and np.isnan(value):
        return True
    s = str(value).strip().lower()
    return s in ['', 'null', 'none', 'nan', 'not mentioned']


def is_match(extracted, ground_truth):
    stopwords = {'a', 'an', 'the', 'with', 'from', 'for', 'of', 'and', 'to', 'in', 'at', 'by'}

    e = set(str(extracted).strip().lower().split()) - stopwords
    g = set(str(ground_truth).strip().lower().split()) - stopwords

    if not e or not g:
        return False

    overlap = e & g
    return len(overlap) / len(g) >= 0.5


# Sanity checks
print('Testing is_null:')
print(f'  None            -> {is_null(None)}')
print(f'  NaN             -> {is_null(float("nan"))}')
print(f'  "Not mentioned" -> {is_null("Not mentioned")}')
print(f'  "Lunch"         -> {is_null("Lunch")}')
print()
print('Testing is_match:')
print(f'  "Microsoft" vs "Microsoft team"  -> {is_match("Microsoft", "Microsoft team")}')
print(f'  "Lunch" vs "lunch buffet"         -> {is_match("Lunch", "lunch buffet")}')
print(f'  "True" vs "Microsoft"             -> {is_match("True", "Microsoft")}')
print(f'  "Amsterdam" vs "Amsterdam"        -> {is_match("Amsterdam", "Amsterdam")}')

Testing is_null:
  None            -> True
  NaN             -> True
  "Not mentioned" -> True
  "Lunch"         -> False

Testing is_match:
  "Microsoft" vs "Microsoft team"  -> True
  "Lunch" vs "lunch buffet"         -> True
  "True" vs "Microsoft"             -> False
  "Amsterdam" vs "Amsterdam"        -> True


---
## Cell 4 — Evaluation Metrics and Logic

Two functions implement the core evaluation logic.

**`compute_metrics(tp, fp, fn, gt_positives)`**  
Computes precision, recall, and F1 from raw counts:
- **Precision** = TP / (TP + FP) — of all values the model extracted, how many were correct?
- **Recall** = TP / gt_positives — of all values that existed in the emails, how many did the model find?
- **F1** = harmonic mean of precision and recall

Recall uses `gt_positives` as its denominator rather than `TP + FN`. The difference is in how wrong extractions are treated. In this pipeline, a wrong extraction (GT has a value, model extracted an incorrect value) is counted as a False Positive only. This keeps FP and FN mutually exclusive and clearly defined: FP means the model extracted something it should not have or extracted the wrong value, and FN means the model returned null for a field that had a value. However, using `TP + FN` as the recall denominator would then exclude wrong extractions from the denominator entirely, making recall appear higher than it actually is. Using `gt_positives` as the denominator corrects this: recall is always computed over the full set of values that existed in the emails, regardless of what the model returned.

Division by zero is handled explicitly. If there are no predictions or no positives, the metric returns 0.0 rather than raising an error.

**`evaluate_configuration(prompt_version, model_name)`**  
Evaluates one prompt-model configuration against the ground truth by looping through every email and every field. Four outcomes are possible for each field-email pair:
- GT has a value, model extracted a matching value → **True Positive (TP)**
- GT has a value, model extracted a non-matching value → **False Positive (FP)**
- GT has a value, model returned null → **False Negative (FN)**
- GT is null, model returned null → **True Negative (TN)** - (Excluded)
- GT is null, model extracted something → **False Positive (FP)**

True negatives are excluded from evaluation to avoid artificially inflating precision and recall by rewarding the model for correctly identifying absent fields.

A test run on one configuration is printed at the end to ensure everything works before the full evaluation.

In [5]:
def compute_metrics(tp, fp, fn, gt_positives):
    precision = tp / (tp + fp)    if (tp + fp) > 0     else 0.0
    recall    = tp / gt_positives  if gt_positives > 0  else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return round(precision, 4), round(recall, 4), round(f1, 4)


def evaluate_configuration(prompt_version, model_name):
    df_config = df_results[
        (df_results['prompt_version'] == prompt_version) &
        (df_results['model_name']     == model_name)
    ]

    field_results = {f: {'tp': 0, 'fp': 0, 'fn': 0} for f in FIELDS}

    for _, gt_row in df_gt_wide.iterrows():
        email_id = gt_row['Email ID']

        ext_row = df_config[df_config['email_id'] == email_id]
        if ext_row.empty:
            continue
        ext_row = ext_row.iloc[0]

        for field in FIELDS:
            gt_value  = gt_row.get(field, GT_NULL)
            ext_value = ext_row.get(field, None)

            gt_is_null  = is_null(gt_value)
            ext_is_null = is_null(ext_value)

            if gt_is_null:
                # GT is null but model extracted something -> FP (hallucination)
                if not ext_is_null:
                    field_results[field]['fp'] += 1
                # Both null -> TN, skip
                continue

            if not ext_is_null:
                if is_match(ext_value, gt_value):
                    field_results[field]['tp'] += 1
                else:
                    field_results[field]['fp'] += 1
            else:
                field_results[field]['fn'] += 1

    return field_results


# Test on one configuration before the full run
test_results = evaluate_configuration('v3', 'gpt-4o')
print('Test — v3 + gpt-4o field counts:')
for field, counts in test_results.items():
    p, r, f1 = compute_metrics(
        counts['tp'], counts['fp'], counts['fn'],
        gt_positives_per_field[field]
    )
    print(f'  {field:<25} TP={counts["tp"]:2d} FP={counts["fp"]:2d} FN={counts["fn"]:2d} | F1={f1:.4f}')

Test — v3 + gpt-4o field counts:
  event_date                TP=50 FP= 0 FN= 0 | F1=1.0000
  event_type                TP=48 FP= 2 FN= 0 | F1=0.9600
  event_time                TP=25 FP=11 FN= 1 | F1=0.6944
  meal_type                 TP=32 FP= 9 FN= 1 | F1=0.8421
  attendee_count            TP=45 FP= 5 FN= 0 | F1=0.9184
  external_guests           TP=27 FP=14 FN= 0 | F1=0.7200
  catering_type             TP=25 FP=23 FN= 0 | F1=0.6667
  office_location           TP=24 FP= 2 FN= 0 | F1=0.9600
  dietary_requirements      TP=19 FP= 8 FN= 0 | F1=0.7308
  room_location             TP=42 FP= 3 FN= 2 | F1=0.9333
  av_equipment              TP= 8 FP= 1 FN= 0 | F1=0.9412
  extra_arrangements        TP= 7 FP= 2 FN=15 | F1=0.4375


---
## Cell 5 — Full Evaluation Across All 9 Configurations

This cell runs the evaluation for every combination of prompt version and model, producing both overall and field-level metrics for all 9 configurations.

For each configuration, the field-level TP, FP, and FN counts are aggregated across all 12 fields to produce overall metrics.

In [6]:
all_overall = []
all_field   = []

for prompt, model in product(PROMPTS, MODELS):
    print(f'Evaluating {prompt} + {model}...', end=' ', flush=True)

    field_results = evaluate_configuration(prompt, model)

    total_tp = sum(r['tp'] for r in field_results.values())
    total_fp = sum(r['fp'] for r in field_results.values())
    total_fn = sum(r['fn'] for r in field_results.values())

    overall_p, overall_r, overall_f1 = compute_metrics(
        total_tp, total_fp, total_fn, total_gt_positives
    )

    print(f'F1={overall_f1:.4f} | P={overall_p:.4f} | R={overall_r:.4f}')

    all_overall.append({
        'prompt_version': prompt,
        'model_name':     model,
        'tp':             total_tp,
        'fp':             total_fp,
        'fn':             total_fn,
        'precision':      overall_p,
        'recall':         overall_r,
        'f1':             overall_f1,
    })

    for field, counts in field_results.items():
        p, r, f1 = compute_metrics(
            counts['tp'], counts['fp'], counts['fn'],
            gt_positives_per_field[field]
        )
        all_field.append({
            'prompt_version': prompt,
            'model_name':     model,
            'field':          field,
            'tp':             counts['tp'],
            'fp':             counts['fp'],
            'fn':             counts['fn'],
            'precision':      p,
            'recall':         r,
            'f1':             f1,
        })

df_overall = pd.DataFrame(all_overall)
df_fields  = pd.DataFrame(all_field)

print()
print(f'Overall results rows:     {len(df_overall)}')
print(f'Field level results rows: {len(df_fields)}')

Evaluating v1 + gpt-4o... F1=0.7944 | P=0.8172 | R=0.7728
Evaluating v1 + gpt-4o-mini... F1=0.7245 | P=0.7493 | R=0.7012
Evaluating v1 + gpt-4.1-nano... F1=0.7110 | P=0.6967 | R=0.7259
Evaluating v2 + gpt-4o... F1=0.7644 | P=0.7447 | R=0.7852
Evaluating v2 + gpt-4o-mini... F1=0.6722 | P=0.6520 | R=0.6938
Evaluating v2 + gpt-4.1-nano... F1=0.6729 | P=0.6346 | R=0.7160
Evaluating v3 + gpt-4o... F1=0.8411 | P=0.8148 | R=0.8691
Evaluating v3 + gpt-4o-mini... F1=0.8343 | P=0.8065 | R=0.8642
Evaluating v3 + gpt-4.1-nano... F1=0.8160 | P=0.7810 | R=0.8543

Overall results rows:     9
Field level results rows: 108


---
## Cell 6 — Save Results and Print Evaluation Matrices

Both result dataframes are saved to CSV here. The 3×3 matrices for F1, precision, and recall are then printed to give a clear side-by-side view of how each configuration performed. The field-level F1 scores for the best performing configuration are also printed to show which fields the pipeline handled well and which it struggled with.

In [ ]:
df_overall.to_csv('Overall_Evaluation.csv', index=False)
df_fields.to_csv('Field_Level_Evaluation.csv', index=False)

print('Saved: Overall_Evaluation.csv')
print('Saved: Field_Level_Evaluation.csv')
print()

print('=' * 55)
print('F1 SCORES — 3x3 configuration matrix')
print('=' * 55)
pivot_f1 = df_overall.pivot(
    index='prompt_version',
    columns='model_name',
    values='f1'
).reindex(index=PROMPTS, columns=MODELS)
print(pivot_f1.to_string())

print()
print('=' * 55)
print('PRECISION — 3x3 configuration matrix')
print('=' * 55)
pivot_p = df_overall.pivot(
    index='prompt_version',
    columns='model_name',
    values='precision'
).reindex(index=PROMPTS, columns=MODELS)
print(pivot_p.to_string())

print()
print('=' * 55)
print('RECALL — 3x3 configuration matrix')
print('=' * 55)
pivot_r = df_overall.pivot(
    index='prompt_version',
    columns='model_name',
    values='recall'
).reindex(index=PROMPTS, columns=MODELS)
print(pivot_r.to_string())

print()
print('=' * 55)
print('FIELD-LEVEL F1 — Prompt 3 + GPT-4o (best config)')
print('=' * 55)
best = df_fields[
    (df_fields['prompt_version'] == 'v3') &
    (df_fields['model_name']     == 'gpt-4o')
][['field', 'precision', 'recall', 'f1']].sort_values('f1', ascending=False)
print(best.to_string(index=False))

Saved: Overall_Evaluation.csv
Saved: Field_Level_Evaluation.csv

F1 SCORES — 3x3 configuration matrix
model_name      gpt-4o  gpt-4o-mini  gpt-4.1-nano
prompt_version                                   
v1              0.7944       0.7245        0.7110
v2              0.7644       0.6722        0.6729
v3              0.8411       0.8343        0.8160

PRECISION — 3x3 configuration matrix
model_name      gpt-4o  gpt-4o-mini  gpt-4.1-nano
prompt_version                                   
v1              0.8172       0.7493        0.6967
v2              0.7447       0.6520        0.6346
v3              0.8148       0.8065        0.7810

RECALL — 3x3 configuration matrix
model_name      gpt-4o  gpt-4o-mini  gpt-4.1-nano
prompt_version                                   
v1              0.7728       0.7012        0.7259
v2              0.7852       0.6938        0.7160
v3              0.8691       0.8642        0.8543

FIELD-LEVEL F1 — Prompt 3 + GPT-4o (best config)
               field  pre

---
## Cell 7 — Mismatch Analysis

This cell provides a detailed breakdown of where the pipeline went wrong for a chosen configuration.

For each field, it identifies and prints:
- **False Positives (FP)** = cases where the model extracted a value but it did not match the ground truth, or where the GT was null but the model extracted something anyway.
- **False Negatives (FN)** = cases where the ground truth had a value but the model returned null.

The configuration to analyse can be changed by updating `prompt_focus` and `model_focus` at the top of the cell. Analysing the worst performing configuration alongside the best gives the clearest picture of where prompt engineering and model capability make the biggest difference.

In [10]:
prompt_focus = 'v3'
model_focus  = 'gpt-4o'

df_config = df_results[
    (df_results['prompt_version'] == prompt_focus) &
    (df_results['model_name']     == model_focus)
]

print(f'Mismatch analysis — {prompt_focus} + {model_focus}')
print('=' * 70)

for field in FIELDS:
    fp_cases = []
    fn_cases = []

    for _, gt_row in df_gt_wide.iterrows():
        email_id  = gt_row['Email ID']
        gt_value  = gt_row.get(field, GT_NULL)
        ext_row   = df_config[df_config['email_id'] == email_id]
        if ext_row.empty:
            continue
        ext_value = ext_row.iloc[0].get(field, None)

        gt_is_null  = is_null(gt_value)
        ext_is_null = is_null(ext_value)

        if gt_is_null:
            if not ext_is_null:
                fp_cases.append((email_id, 'Not mentioned', str(ext_value)))
            continue

        if not ext_is_null and not is_match(ext_value, gt_value):
            fp_cases.append((email_id, str(gt_value), str(ext_value)))
        elif ext_is_null:
            fn_cases.append((email_id, str(gt_value)))

    if fp_cases or fn_cases:
        print(f'\n── {field} (FP={len(fp_cases)}, FN={len(fn_cases)}) ──')

        if fp_cases:
            print('  False Positives (wrong value or hallucinated):')
            for eid, gt, ext in fp_cases:
                print(f"    Email {eid:2d} | GT: '{gt[:50]}' | Extracted: '{ext[:50]}'")

        if fn_cases:
            print('  False Negatives (missed extraction):')
            for eid, gt in fn_cases:
                print(f"    Email {eid:2d} | GT: '{gt[:50]}' | Extracted: null")

Mismatch analysis — v3 + gpt-4o

── event_type (FP=2, FN=0) ──
  False Positives (wrong value or hallucinated):
    Email  3 | GT: 'Meeting' | Extracted: 'Partner event with SAP'
    Email 31 | GT: 'strategy catch up' | Extracted: 'Strategy catch-up'

── event_time (FP=11, FN=1) ──
  False Positives (wrong value or hallucinated):
    Email  2 | GT: '13:00:00' | Extracted: 'Starting around 1:00 PM, entire afternoon'
    Email  7 | GT: '09:00:00' | Extracted: '9:00 AM'
    Email 21 | GT: 'Not mentioned' | Extracted: '8:00 AM'
    Email 22 | GT: '2PM to 5PM' | Extracted: '2 PM to 5 PM'
    Email 28 | GT: '9:00AM' | Extracted: '9:00 AM'
    Email 30 | GT: '1:00PM to 5:00PM' | Extracted: '1:00 PM to 5:00 PM'
    Email 33 | GT: '08:30 till noon' | Extracted: '8:30 AM until noon'
    Email 34 | GT: '2PM to 5PM' | Extracted: '2 PM to 5 PM'
    Email 38 | GT: 'start 2:00PM' | Extracted: '2:00 PM'
    Email 45 | GT: 'starting from 16:00 ending with a dinner at 18:30' | Extracted: '16:00 to 18:30